# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# Unit of analysis: ONE ROW = one content page (content_hash_id), for one client
# (client_hash_id), measured over a defined trailing window.
#
# Table: fact_content_daily_performance (joined with dim_content for metadata)
#
# Time window: I will use a mid-panel month, month=2026-03, for feature
# development (as instructed - the freshest month is reserved as a sealed
# test window and must not be used to build label logic).
#
# This matches my Lane 2 question: which pages should a review team look at
# first for refresh/expansion/protection - the grain (one page) is exactly
# what a reviewer would open and act on.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
# The Data Contract (5 answers):
#
# 1. What one row means: one content page, for one client, on one day -
#    aggregated over my chosen feature window (prior 90 days).
#
# 2. Table(s) used: fact_content_daily_performance (daily facts) joined to
#    dim_content (content metadata) on content_hash_id.
#
# 3. Time window: features built from month=2026-03 (mid-panel, safe for
#    label development). The final month (2026-06) is reserved as a sealed
#    test window later - never used to shape label logic now.
#
# 4. Label/proxy: a page is a "review candidate" if it shows declining
#    trend/visibility with meaningful demand (impressions) in the feature
#    window - this is a proxy, not a guaranteed cause of future recovery.
#
# 5. Deliberately excluded: I exclude any FlyRank product decision fields
#    (health_score, priority_score, action_type) - these are not in the
#    warehouse release anyway, but I confirm I will never reconstruct and
#    feed them back in as a feature, since that would just teach the model
#    to copy an existing rule instead of finding real signal.x

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
import os
from google.colab import userdata
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

# Query 1: grain check - is one row really "one page, one client, one day"?
q1 = con.sql("""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain check (empty = grain confirmed correct):")
print(q1)

# Query 2: row count and date span for the slice
q2 = con.sql("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("\nRow count and date span:")
print(q2)

# Query 3: availability - filter with IS TRUE
q3 = con.sql("""
    SELECT COUNT(*) as rows_with_ga4
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()
print("\nRows with GA4 data available:")
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain check (empty = grain confirmed correct):
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, row_count]
Index: []

Row count and date span:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows with GA4 data available:
   rows_with_ga4
0         413966


In [11]:
cols = con.sql("""
    DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()
print(cols)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [12]:
# Aggregate to one row per page for the month (my unit of analysis)
features = con.sql("""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions_month,
        SUM(scroll_events) AS scroll_events_month
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

# Safe CTR calc
features['ctr'] = features['clicks_month'] / features['impressions_month']

print(f"Feature frame: {len(features)} pages, one row per page for March 2026")
features.head()

# Five features, max - each with "knowable at decision moment because..."
#
# 1. impressions_month - knowable because GSC logs impressions daily; by the
#    end of the month, this total is fully observed history, not a forecast.
# 2. clicks_month - same reason: fully observed GSC history for the window.
# 3. avg_position - knowable: average ranking position over the window is
#    measured after the fact, from search console data already collected.
# 4. sessions_month - knowable: GA4 sessions are logged as they happen,
#    available in full by the time the window closes.
# 5. ctr (clicks_month / impressions_month) - derived only from the two
#    features above, calculated strictly within the same feature window.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176738 pages, one row per page for March 2026


,content_hash_id,impressions_month,clicks_month,avg_position,sessions_month,scroll_events_month,ctr
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,0.001754
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,0.000000
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,0.000000
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,1.0,0.004222
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,0.005776


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Define a simple proxy label: "review candidate" = low CTR despite decent visibility
features_clean = features[features['impressions_month'] >= 100].copy()
features_clean['label'] = (features_clean['ctr'] < features_clean['ctr'].median()).astype(int)

# HONEST quick score: using position and impressions only (no leakage)
X_honest = features_clean[['avg_position', 'impressions_month']]
y = features_clean['label']
model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
auc_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (position + impressions only): {auc_honest:.3f}")

# THE TRAP: deliberately add a label-derived column - ctr itself, which is
# literally what the label was thresholded from
features_clean['ctr_leaked'] = features_clean['ctr']
X_leaked = features_clean[['avg_position', 'impressions_month', 'ctr_leaked']]
model_leaked = LogisticRegression(max_iter=1000).fit(X_leaked, y)
auc_leaked = roc_auc_score(y, model_leaked.predict_proba(X_leaked)[:, 1])
print(f"LEAKED AUC (with ctr_leaked added): {auc_leaked:.3f}  <- jumps toward perfect!")

# Delete the leaked column and keep the honest number
features_clean = features_clean.drop(columns=['ctr_leaked'])
print(f"\nLesson: adding a column derived directly from the label made the score jump")
print(f"from {auc_honest:.3f} to {auc_leaked:.3f} - not because the model got smarter,")
print(f"but because it was literally given the answer. Removed. Honest AUC = {auc_honest:.3f}")

Honest AUC (position + impressions only): 0.696
LEAKED AUC (with ctr_leaked added): 0.836  <- jumps toward perfect!

Lesson: adding a column derived directly from the label made the score jump
from 0.696 to 0.836 - not because the model got smarter,
but because it was literally given the answer. Removed. Honest AUC = 0.696


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
# Data limits (named honestly):
#
# 1. Unbalanced panel: different clients have different amounts of history -
#    some clients have 12+ months, others much less. A page's "trend" looks
#    different depending on how much history exists for its client.
#
# 2. GSC-only early rows: before a client's GA4 tracking started, rows only
#    have search (GSC) data - ga4_data_available = FALSE. Treating these as
#    "zero traffic" instead of "not tracked yet" would be a mistake.
#
# 3. Window overlap risk: if my feature window and target window ever share
#    days, that's leakage - I must keep them strictly separated (e.g., prior
#    90 days for features, a separate later window for any future label).
#
# 4. This slice cannot prove causation: even a strong pattern only tells me
#    what is associated with review-worthy pages, not that fixing them will
#    cause recovery - that would need a controlled experiment.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.